[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/225/blob/main/10_pandas_sql/10_pandas_sql_practice_quiz.ipynb)

# 10 — pandas and SQL: Practice Quiz

These questions cover the full module: `df.query()`, SQLite setup, SELECT/WHERE, GROUP BY/HAVING, JOIN, and subqueries. All questions use the Gapminder dataset split into the same `countries` and `measurements` tables used throughout the module.

In [ ]:
import pandas as pd
import sqlite3

df = pd.read_parquet("gapminder.parquet")

countries = df[["country", "continent"]].drop_duplicates().reset_index(drop=True)
measurements = df[["country", "year", "lifeExp", "pop", "gdpPercap"]].copy()

conn = sqlite3.connect(":memory:")
countries.to_sql("countries", conn, index=False, if_exists="replace")
measurements.to_sql("measurements", conn, index=False, if_exists="replace")

print("Ready.")

---
### Question 1

Which `df.query()` call correctly finds all rows where `gdpPercap` is above a threshold stored in the variable `cutoff`?

A. `df.query("gdpPercap > cutoff")`  
B. `df.query("gdpPercap > @cutoff")`  
C. `df.query(f"gdpPercap > {cutoff}")`  
D. Both B and C are correct

**Your answer:**

In [ ]:
answer = ""

In [ ]:
#@title Solution
# Answer: D
# @cutoff is the canonical approach: query() looks up the variable in the surrounding
# Python scope. An f-string also works (it embeds the numeric value before query() sees
# the string), but it has a subtle risk: if the value happens to match a column name
# or contains special characters, the result can be wrong or raise an error.
# Option A is wrong: query() would treat 'cutoff' as a column name, not a variable.
cutoff = 10000
result_at = df.query("gdpPercap > @cutoff")
result_fstring = df.query(f"gdpPercap > {cutoff}")
print("Same result:", result_at.shape == result_fstring.shape)

---
### Question 2

What does `SELECT DISTINCT continent FROM measurements` return?

A. An error, because `continent` is not in the `measurements` table  
B. All 1,704 rows with their continent values  
C. 5 rows, one per unique continent  
D. 142 rows, one per unique country

**Your answer:**

In [ ]:
answer = ""

In [ ]:
#@title Solution
# Answer: A
# The measurements table has country, year, lifeExp, pop, gdpPercap -- no continent.
# continent lives in the countries table. To get continent from measurements, you need
# a JOIN with the countries table.
try:
    pd.read_sql("SELECT DISTINCT continent FROM measurements", conn)
except Exception as e:
    print(f"Error: {e}")

# The correct version:
pd.read_sql("SELECT DISTINCT continent FROM countries", conn)

---
### Question 3

In SQL, the equality operator for filtering is:

A. `==` (same as Python)  
B. `=` (single equals sign)  
C. `.eq()` (method call)  
D. `IS` (keyword)

**Your answer:**

In [ ]:
answer = ""

In [ ]:
#@title Solution
# Answer: B
# SQL uses = for equality comparisons in WHERE, not ==.
# IS is used only for NULL comparisons: WHERE continent IS NULL.
pd.read_sql("""
    SELECT country, year, lifeExp
    FROM measurements
    WHERE year = 2007
    ORDER BY lifeExp DESC
    LIMIT 5
""", conn)

---
### Question 4

Which SQL clause correctly filters on an aggregate function (e.g., keeping only groups where the average exceeds a threshold)?

A. `WHERE AVG(lifeExp) > 70`  
B. `HAVING AVG(lifeExp) > 70`  
C. `FILTER AVG(lifeExp) > 70`  
D. `SELECT AVG(lifeExp) > 70`

**Your answer:**

In [ ]:
answer = ""

In [ ]:
#@title Solution
# Answer: B
# HAVING runs after GROUP BY and can reference aggregate functions.
# WHERE runs before grouping and cannot reference aggregates (raises an error).
pd.read_sql("""
    SELECT c.continent, ROUND(AVG(m.lifeExp), 1) AS avg_life
    FROM measurements AS m
    JOIN countries AS c ON m.country = c.country
    WHERE m.year = 2007
    GROUP BY c.continent
    HAVING AVG(m.lifeExp) > 70
    ORDER BY avg_life DESC
""", conn)

---
### Question 5

What is the correct SQL clause order for a query that uses SELECT, FROM, WHERE, GROUP BY, HAVING, ORDER BY, and LIMIT?

A. SELECT → FROM → WHERE → GROUP BY → HAVING → ORDER BY → LIMIT  
B. FROM → SELECT → WHERE → GROUP BY → HAVING → ORDER BY → LIMIT  
C. SELECT → FROM → GROUP BY → WHERE → HAVING → ORDER BY → LIMIT  
D. SELECT → WHERE → FROM → GROUP BY → HAVING → ORDER BY → LIMIT

**Your answer:**

In [ ]:
answer = ""

In [ ]:
#@title Solution
# Answer: A
# The clause order in SQL is fixed: SELECT ... FROM ... WHERE ... GROUP BY ...
# HAVING ... ORDER BY ... LIMIT. Not every query uses every clause, but when
# they appear, they must be in this order.
print("Clause order: SELECT → FROM → WHERE → GROUP BY → HAVING → ORDER BY → LIMIT")

---
### Question 6

You run this query:
```sql
SELECT m.country, c.continent, m.year, m.lifeExp
FROM measurements AS m
INNER JOIN countries AS c ON m.country = c.country
WHERE m.country = 'Narnia'
```
Narnia exists in `measurements` but not in `countries`. What does the query return?

A. One row with `continent = NULL`  
B. Zero rows (Narnia is dropped because INNER JOIN requires a match in both tables)  
C. An error because the JOIN finds no matching row  
D. All rows from `measurements` where country = 'Narnia', with continent left blank

**Your answer:**

In [ ]:
answer = ""

In [ ]:
#@title Solution
# Answer: B
# INNER JOIN keeps only rows with a match in BOTH tables. Since Narnia has no
# entry in countries, it is dropped entirely. Use LEFT JOIN to keep unmatched rows.
import pandas as pd
fake = pd.DataFrame([{"country": "Narnia", "year": 2007,
                       "lifeExp": 999.0, "pop": 1000, "gdpPercap": 50000.0}])
import pandas as pd
m_with_fake = pd.concat([measurements, fake], ignore_index=True)
m_with_fake.to_sql("measurements", conn, index=False, if_exists="replace")

result_inner = pd.read_sql("""
    SELECT m.country, c.continent FROM measurements AS m
    INNER JOIN countries AS c ON m.country = c.country
    WHERE m.country = 'Narnia'
""", conn)
print("INNER JOIN rows:", len(result_inner))

result_left = pd.read_sql("""
    SELECT m.country, c.continent FROM measurements AS m
    LEFT JOIN countries AS c ON m.country = c.country
    WHERE m.country = 'Narnia'
""", conn)
print("LEFT JOIN rows:", len(result_left))
print(result_left)

# Restore clean measurements
measurements.to_sql("measurements", conn, index=False, if_exists="replace")

---
### Question 7

What is the difference between `COUNT(*)` and `COUNT(col)` in a GROUP BY query?

A. `COUNT(*)` counts all rows in the group including NULLs; `COUNT(col)` counts only non-NULL values in that column  
B. `COUNT(*)` counts distinct values; `COUNT(col)` counts total rows  
C. They are identical; the `*` and column name are interchangeable  
D. `COUNT(*)` counts rows; `COUNT(col)` counts the number of columns in the table

**Your answer:**

In [ ]:
answer = ""

In [ ]:
#@title Solution
# Answer: A
# COUNT(*) counts every row in the group regardless of NULL values.
# COUNT(col) counts only rows where that column is not NULL.
# In a clean dataset like Gapminder they agree; they differ when nulls are present.
# COUNT(DISTINCT col) is a third form: counts unique non-NULL values.
pd.read_sql("""
    SELECT c.continent,
           COUNT(*) AS count_star,
           COUNT(m.lifeExp) AS count_col,
           COUNT(DISTINCT m.country) AS count_distinct
    FROM measurements AS m
    JOIN countries AS c ON m.country = c.country
    GROUP BY c.continent
""", conn)

---
### Question 8

Write a SQL query that returns the 5 countries with the lowest GDP per capita in the year 2007. Return `country` and `gdpPercap` rounded to 2 decimal places.

In [ ]:
# your code here

In [ ]:
#@title Solution
pd.read_sql("""
    SELECT country, ROUND(gdpPercap, 2) AS gdpPercap
    FROM measurements
    WHERE year = 2007
    ORDER BY gdpPercap ASC
    LIMIT 5
""", conn)

---
### Question 9

Write a SQL query using `GROUP BY` and a `JOIN` that returns, for each continent, the number of distinct countries and the maximum life expectancy in 2007. Name the columns `n_countries` and `max_life`. Sort by `max_life` descending.

In [ ]:
# your code here

In [ ]:
#@title Solution
pd.read_sql("""
    SELECT c.continent,
           COUNT(DISTINCT m.country) AS n_countries,
           ROUND(MAX(m.lifeExp), 1) AS max_life
    FROM measurements AS m
    JOIN countries AS c ON m.country = c.country
    WHERE m.year = 2007
    GROUP BY c.continent
    ORDER BY max_life DESC
""", conn)

---
### Question 10

Write a SQL query using `HAVING` to find all countries whose average GDP per capita across ALL years exceeded $15,000. Return `country` and the average GDP rounded to 0 decimal places, sorted by average GDP descending. How many countries qualify?

In [ ]:
# your code here

In [ ]:
#@title Solution
result = pd.read_sql("""
    SELECT country,
           ROUND(AVG(gdpPercap), 0) AS avg_gdp
    FROM measurements
    GROUP BY country
    HAVING AVG(gdpPercap) > 15000
    ORDER BY avg_gdp DESC
""", conn)
print(f"{len(result)} countries qualify")
result

---
### Question 11

Write a SQL subquery to find all countries whose 2007 GDP per capita exceeded the global average 2007 GDP per capita. Return `country` and `gdpPercap`, sorted descending. How many countries qualify?

In [ ]:
# your code here

In [ ]:
#@title Solution
result = pd.read_sql("""
    SELECT country, ROUND(gdpPercap, 0) AS gdpPercap
    FROM measurements
    WHERE year = 2007
      AND gdpPercap > (
          SELECT AVG(gdpPercap)
          FROM measurements
          WHERE year = 2007
      )
    ORDER BY gdpPercap DESC
""", conn)
print(f"{len(result)} countries above global 2007 average GDP")
result.head(10)

---
### Question 12

Use the two-step workflow: SQL for aggregation, pandas for visualization.

Write a SQL query that returns, for each continent and year, the total population (in millions, rounded to 1 decimal place). Then use the result to plot a line chart showing total population over time by continent.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", context="notebook")

# your code here

In [ ]:
#@title Solution
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", context="notebook")

# Step 1: SQL aggregation
pop_summary = pd.read_sql("""
    SELECT c.continent,
           m.year,
           ROUND(SUM(m.pop) / 1000000.0, 1) AS total_pop_millions
    FROM measurements AS m
    JOIN countries AS c ON m.country = c.country
    GROUP BY c.continent, m.year
    ORDER BY c.continent, m.year
""", conn)

# Step 2: pandas visualization
fig, ax = plt.subplots(figsize=(9, 4))
for continent, group in pop_summary.groupby("continent"):
    ax.plot(group["year"], group["total_pop_millions"], marker="o", label=continent)
ax.set_title("Total population by continent, 1952–2007 (millions)")
ax.set_xlabel("Year")
ax.set_ylabel("Population (millions)")
ax.legend(title="Continent")